In [ ]:

# V28 224px slot cache. CPU only. Never writes submission.csv.
# Output: /kaggle/working/v28_cache/{x_f16.dat,mask.dat,y.npy,uids.txt,manifest.json}
from __future__ import annotations

import gc
import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom

cv2.setNumThreads(1)
T0 = time.time()
IMG = 224
CROP_MM = 160.0
N_SLICE = 3
N_SLOT = 3
SLOTS = [("Sagittal", True), ("Coronal", True), ("Axial", True)]
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
ALIVE = 8.2 * 3600
OUT = Path("/kaggle/working/v28_cache")
OUT.mkdir(parents=True, exist_ok=True)


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def find_dir(*names):
    root = Path("/kaggle/input")
    hits = []
    if not root.is_dir():
        return None
    for p in root.rglob("*"):
        if p.is_dir() and p.name in names:
            hits.append(p)
    return hits[0] if hits else None


def find_file(name):
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    for p in root.rglob(name):
        if p.is_file():
            return p
    return None


COMP = find_dir("rsna-knee-abnormality-detection")
assert COMP is not None, "competition data missing"
log(f"comp={COMP}")


def load_soft_labels():
    files = []
    for name in ("report_labels_v2.csv", "llm_labels_v2.csv", "labels_llm_gpt56sol.csv"):
        p = find_file(name)
        if p is None:
            log(f"label missing: {name}")
            continue
        df = pd.read_csv(p, dtype={"StudyInstanceUID": str})
        keep = ["StudyInstanceUID"] + [c for c in TARGETS if c in df.columns]
        files.append(df[keep].set_index("StudyInstanceUID"))
        log(f"label {name} rows={len(df)}")
    assert files, "no label tables"
    idx = files[0].index
    for f in files[1:]:
        idx = idx.union(f.index)
    out = pd.DataFrame(index=idx)
    for t in TARGETS:
        cols = [f[t] for f in files if t in f.columns]
        stacked = pd.concat(cols, axis=1)
        out[t] = stacked.mean(axis=1)
    train = pd.read_csv(COMP / "train.csv", dtype={"StudyInstanceUID": str})
    gold = train.dropna(subset=TARGETS, how="all")
    if len(gold):
        g = gold.set_index("StudyInstanceUID")[TARGETS].apply(pd.to_numeric, errors="coerce")
        out.loc[g.index.intersection(out.index), TARGETS] = g.loc[g.index.intersection(out.index)]
        log(f"gold overwrite {len(g)}")
    return out.reset_index().rename(columns={"index": "StudyInstanceUID"})


def physical_order(sdir: Path):
    keyed = []
    for f in sdir.glob("*.dcm"):
        ds = None
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            iop = np.array(ds.ImageOrientationPatient, dtype=np.float64).reshape(2, 3)
            normal = np.cross(iop[0], iop[1])
            ipp = np.array(ds.ImagePositionPatient, dtype=np.float64)
            key = float(np.dot(ipp, normal))
        except Exception:
            try:
                key = float(getattr(ds, "InstanceNumber", 0) or 0) if ds is not None else 0.0
            except Exception:
                key = 0.0
        keyed.append((key, f))
    keyed.sort()
    return [f for _, f in keyed]


def read_crop(path: Path):
    try:
        ds = pydicom.dcmread(str(path))
        arr = ds.pixel_array.astype(np.float32)
        arr *= float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
        arr += float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / max(ps, 1e-3) / 2))
    cy, cx = arr.shape[0] // 2, arr.shape[1] // 2
    y0, y1 = max(0, cy - half), min(arr.shape[0], cy + half)
    x0, x1 = max(0, cx - half), min(arr.shape[1], cx + half)
    crop = arr[y0:y1, x0:x1]
    if crop.size == 0:
        return None
    lo, hi = np.percentile(crop, [1, 99])
    crop = np.clip((crop - lo) / max(hi - lo, 1e-6), 0, 1)
    img = cv2.resize(crop, (IMG, IMG), interpolation=cv2.INTER_AREA)
    return img


def pick_fs(series_df, plane):
    sub = series_df[series_df.Anatomical_Plane == plane]
    if sub.empty:
        return None
    scored = sub.copy()
    if "Fat_Suppression" in scored.columns:
        scored["_fs"] = pd.to_numeric(scored.Fat_Suppression, errors="coerce").fillna(0)
    elif "Fluid_Sensitive" in scored.columns:
        scored["_fs"] = pd.to_numeric(scored.Fluid_Sensitive, errors="coerce").fillna(0)
    else:
        scored["_fs"] = 0
    return scored.sort_values("_fs", ascending=False).iloc[0]


def build_study(uid, recs, series_root):
    x = np.zeros((N_SLOT, N_SLICE, IMG, IMG), np.float32)
    m = np.zeros(N_SLOT, np.uint8)
    rows = pd.DataFrame(recs)
    for i, (plane, _fs) in enumerate(SLOTS):
        rec = pick_fs(rows, plane)
        if rec is None:
            continue
        sdir = series_root / uid / str(rec.SeriesInstanceUID)
        if not sdir.is_dir():
            continue
        files = physical_order(sdir)
        if not files:
            continue
        lo, hi = 0.20, 0.80
        i0 = int(round(lo * (len(files) - 1)))
        i1 = int(round(hi * (len(files) - 1)))
        mid = (i0 + i1) // 2
        picks = [max(0, mid - 1), mid, min(len(files) - 1, mid + 1)]
        for j, p in enumerate(picks[:N_SLICE]):
            img = read_crop(files[p])
            if img is not None:
                x[i, j] = img
                m[i] = 1
    return x, m


def main():
    labels = load_soft_labels()
    train_csv = pd.read_csv(COMP / "train.csv", dtype={"StudyInstanceUID": str})
    series = pd.read_csv(COMP / "train_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
    uids = [u for u in train_csv.StudyInstanceUID.astype(str) if u in set(labels.StudyInstanceUID)]
    labels = labels.set_index("StudyInstanceUID").loc[uids]
    y = np.clip(labels[TARGETS].to_numpy(np.float32), 0, 1)
    recs = {s: g.to_dict("records") for s, g in series.groupby("StudyInstanceUID")}
    series_root = COMP / "train_series"
    n = len(uids)
    log(f"n={n}")
    x_path = OUT / "x_f16.dat"
    m_path = OUT / "mask.dat"
    prog_path = OUT / "progress.json"
    start = 0
    if prog_path.is_file() and x_path.is_file() and m_path.is_file():
        try:
            start = int(json.loads(prog_path.read_text())["done"])
            log(f"resume cache at {start}")
        except Exception:
            start = 0
    xmm = np.memmap(x_path, dtype=np.float16, mode="r+" if start else "w+", shape=(n, N_SLOT, N_SLICE, IMG, IMG))
    mmm = np.memmap(m_path, dtype=np.uint8, mode="r+" if start else "w+", shape=(n, N_SLOT))
    n_ok = 0
    for i, uid in enumerate(uids):
        if i < start:
            continue
        if time.time() - T0 > ALIVE:
            prog_path.write_text(json.dumps({"done": i, "n": n, "status": "alive_stop"}) + "\n")
            xmm.flush()
            mmm.flush()
            log(f"alive stop at {i}/{n}")
            return
        xx, mm = build_study(uid, recs.get(uid, []), series_root)
        xmm[i] = xx.astype(np.float16)
        mmm[i] = mm
        n_ok += int(mm.sum() > 0)
        if i % 50 == 0:
            xmm.flush()
            mmm.flush()
            prog_path.write_text(json.dumps({"done": i + 1, "n": n, "n_ok": n_ok}) + "\n")
            log(f"{i+1}/{n} ok_slots_last={int(mm.sum())}")
        if i % 200 == 0:
            gc.collect()
    xmm.flush()
    mmm.flush()
    np.save(OUT / "y.npy", y)
    (OUT / "uids.txt").write_text("\n".join(uids) + "\n", encoding="utf-8")
    man = {
        "n": n,
        "x_shape": [n, N_SLOT, N_SLICE, IMG, IMG],
        "dtype": "float16",
        "img": IMG,
        "crop_mm": CROP_MM,
        "recipe": "v28-224-3slice-3fs-planes",
        "targets": TARGETS,
        "n_ok_last_pass": n_ok,
        "seconds": time.time() - T0,
        "status": "complete",
    }
    (OUT / "manifest.json").write_text(json.dumps(man, indent=2) + "\n")
    prog_path.write_text(json.dumps({"done": n, "n": n, "status": "complete"}) + "\n")
    log(json.dumps(man))
    sub = Path("/kaggle/working/submission.csv")
    if sub.exists():
        sub.unlink()


main()
